<a href="https://colab.research.google.com/github/rodricard/Proyecto-de-IA/blob/main/Proyecto_de_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
langchain \
langchain-community \
langchain-mistralai \
faiss-cpu \
sentence-transformers \
pypdf \
pandas \
numpy

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from langchain_mistralai import ChatMistralAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.text_splitter import CharacterTextSplitter

print("✅ Librerías importadas correctamente")

In [ ]:
# CONFIGURAR API KEY DE MISTRAL
MISTRAL_API_KEY = "QxpqzC8TTViLaDlJeIrYk29vEsHRx2vI"
os.environ["MISTRAL_API_KEY"] = MISTRAL_API_KEY

print("✅ API Key configurada")

In [ ]:
# CARGAR DATASET
df = pd.read_csv('/content/ventas.csv.csv', encoding='latin1')

print("✅ Dataset cargado correctamente")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

display(df.head())

In [ ]:
# NORMALIZACIÓN DEL DATASET
df.columns = df.columns.str.strip().str.lower()

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

scaler = MinMaxScaler()

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("✅ Dataset normalizado")
display(df.head())

In [ ]:
# GUARDAR DATASET NORMALIZADO
df.to_csv('/content/ventas_normalizado.csv', index=False)

print("✅ Archivo guardado: ventas_normalizado.csv")

In [ ]:
# CREAR CORPUS PARA EL AGENTE

# Dataset original para respuestas reales
df_original = pd.read_csv('/content/ventas.csv.csv', encoding='latin1')
df_original.columns = df_original.columns.str.strip().str.lower()

# Convertir filas en texto
documentos = []

for _, row in df_original.iterrows():
    contenido = " | ".join([f"{col}: {row[col]}" for col in df_original.columns])
    documentos.append(Document(page_content=contenido))

print(f"✅ Documentos creados: {len(documentos)}")

In [ ]:
# DIVIDIR TEXTO Y CREAR EMBEDDINGS

splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

docs = splitter.split_documents(documentos)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)

print("✅ VectorStore FAISS creado correctamente")

/tmp/ipykernel_5447/3145879885.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ VectorStore FAISS creado correctamente


In [ ]:
# CREAR AGENTE CON MISTRAL + LANGCHAIN

llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0,
    api_key=MISTRAL_API_KEY
)

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""Eres un asistente que responde preguntas sobre un dataset de ventas.

Usa SOLAMENTE la información del contexto.

Contexto:
{context}

Pregunta:
{question}

Respuesta corta y clara:
"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template}
)

print("✅ Agente creado correctamente")

In [ ]:
# TEST DE CONEXIÓN CON MISTRAL

try:
    respuesta = llm.invoke("Hola")
    print("✅ Conexión exitosa con Mistral")
    print(respuesta.content)
except Exception as e:
    print("❌ Error de conexión")
    print(e)

In [ ]:
# PREGUNTAS AL AGENTE

preguntas = [
    "¿Cuál es el producto más vendido?",
    "¿Qué cliente realizó más compras?",
    "¿Cuál es el monto de venta más alto?",
    "¿Qué tipo de producto aparece más veces?"
]

for i, pregunta in enumerate(preguntas, start=1):
    print("\n" + "="*60)
    print(f"PREGUNTA {i}: {pregunta}")
    print("-"*60)

    try:
        respuesta = qa_chain.invoke({"query": pregunta})
        print(respuesta["result"])
    except Exception as e:
        print("Error:", e)
